In [ ]:
# =============================================================================
# 03_trends_mann_kendall.ipynb  —  HPC version
# Per-pixel Mann-Kendall trend + Sen's slope + p-value for 6 variables,
# run per season (JF, MAM, JJAS, OND).
#
# Inputs : data/processed/ and data/derived/ (.nc stacks)
# Outputs: data/trends/
#            <VAR>_<SEASON>_trend.nc    (3 bands: slope, p, tau)
#            tifs/<VAR>/<VAR>_<SEASON>_slope.tif
#            tifs/<VAR>/<VAR>_<SEASON>_pvalue.tif
#            tifs/<VAR>/<VAR>_<SEASON>_significant.tif   (0/1 mask)
# =============================================================================

import os, gc, time, warnings
from pathlib import Path
from multiprocessing import Pool, cpu_count

import numpy as np
import pandas as pd
import xarray as xr
import rioxarray as rxr
import pymannkendall as mk
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

# ---------------- Paths ----------------
REPO_ROOT  = Path('/scratch/lustre/users/fngari/John/SOIL-MOISTURE-PREDICTION')
PROC_DIR   = REPO_ROOT / 'data' / 'processed'
DERIVED_DIR= REPO_ROOT / 'data' / 'derived'
TREND_DIR  = REPO_ROOT / 'data' / 'trends'
TIF_DIR    = TREND_DIR / 'tifs'
LOG_DIR    = REPO_ROOT / 'notebooks' / 'logs'

TREND_DIR.mkdir(parents=True, exist_ok=True)
TIF_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

# ---------------- Config ----------------
CRS        = 'EPSG:21037'
NODATA     = -9999
SEASONS    = ['JF', 'MAM', 'JJAS', 'OND']
N_MIN      = 20       # min valid seasons required to run MK
ALPHA      = 0.05     # significance threshold for p-value
N_WORKERS  = 16       # parallel workers (matches srun --cpus-per-task)

# Variable -> (stack path, data variable name)
VARIABLES = {
    'NDVI':             (PROC_DIR    / 'NDVI_stack_qa.nc',         'NDVI'),
    'FVC':              (DERIVED_DIR / 'FVC_stack.nc',             'FVC'),
    'FOREST_DENSITY':   (DERIVED_DIR / 'FOREST_DENSITY_stack.nc',  'FOREST_DENSITY'),
    'SM_L2':            (PROC_DIR    / 'SM_L2_stack.nc',           'SM_L2'),
    'SPEI_3':           (DERIVED_DIR / 'SPEI_3_stack.nc',          'SPEI_3'),
    'PRECIP':           (PROC_DIR    / 'PRECIP_stack.nc',          'PRECIP'),
}

print(f"Workers   : {N_WORKERS}")
print(f"Variables : {list(VARIABLES.keys())}")
print(f"Seasons   : {SEASONS}")
print(f"N_min     : {N_MIN}")

In [ ]:
# =============================================================================
# Worker: runs MK + Sen on a single pixel's time series.
# Returns (slope, p_value, tau) or (nan, nan, nan) if insufficient data.
#
# This function must be at module level (not inside another function)
# for multiprocessing.Pool to pickle it.
# =============================================================================

def _mk_one_pixel(series, n_min):
    """series: 1D float32/float64 array with NaN for missing."""
    valid = series[np.isfinite(series)]
    if valid.size < n_min:
        return (np.nan, np.nan, np.nan)
    if np.all(valid == valid[0]):    # constant series → no trend
        return (0.0, 1.0, 0.0)
    try:
        res = mk.original_test(valid)
        # res.slope = Sen's slope; res.p = p-value; res.Tau = Kendall's tau
        return (float(res.slope), float(res.p), float(res.Tau))
    except Exception:
        return (np.nan, np.nan, np.nan)

In [ ]:
# =============================================================================
# Process MK row-by-row, across all rows in parallel.
# Each worker handles one row of pixels and returns (slope, p, tau) arrays.
# =============================================================================

# Global for workers (set via Pool initializer)
_G_SERIES   = None
_G_NMIN     = None

def _init_worker(series_flat, n_min):
    global _G_SERIES, _G_NMIN
    _G_SERIES = series_flat
    _G_NMIN   = n_min

def _process_row(j):
    """j = row index. Returns three (nx,) arrays: slope, p, tau."""
    nx = _G_SERIES.shape[1]
    slope = np.full(nx, np.nan, dtype='float32')
    pval  = np.full(nx, np.nan, dtype='float32')
    tau   = np.full(nx, np.nan, dtype='float32')

    for k in range(nx):
        s, p, t = _mk_one_pixel(_G_SERIES[:, j, k], _G_NMIN)
        slope[k] = s
        pval[k]  = p
        tau[k]   = t

    return slope, pval, tau


def run_mk_parallel(da, n_min=N_MIN, n_workers=N_WORKERS):
    """
    da: xr.DataArray with dims (time, y, x), one season per year.
    Returns: dict with 'slope', 'p', 'tau' as (y, x) float32 arrays.
    """
    nt, ny, nx = da.shape
    series_flat = da.values.astype('float32')   # (nt, ny, nx)

    slope = np.full((ny, nx), np.nan, dtype='float32')
    pval  = np.full((ny, nx), np.nan, dtype='float32')
    tau   = np.full((ny, nx), np.nan, dtype='float32')

    rows = list(range(ny))

    with Pool(processes=n_workers,
              initializer=_init_worker,
              initargs=(series_flat, n_min)) as pool:
        for j, (s_row, p_row, t_row) in enumerate(
                tqdm(pool.imap(_process_row, rows, chunksize=16),
                     total=ny, desc='MK rows')):
            slope[j] = s_row
            pval[j]  = p_row
            tau[j]   = t_row

    return {'slope': slope, 'p': pval, 'tau': tau}

In [ ]:
# =============================================================================
# Main loop: for each variable, for each season, run MK on the 31-year series.
# Saves per-variable-per-season NetCDF + GeoTIFFs.
# =============================================================================

def save_trend_outputs(var_name, season, results,
                       y_coord, x_coord, transform):
    """Write 3-band NetCDF + 3 GeoTIFFs for this var/season."""
    slope = results['slope']
    p     = results['p']
    tau   = results['tau']

    # --- NetCDF ---
    da_stack = xr.Dataset(
        {
            'slope': (('y', 'x'), slope, {'units': 'per year',
                                          'description': "Sen's slope"}),
            'p':     (('y', 'x'), p,     {'units': 'probability',
                                          'description': 'MK p-value'}),
            'tau':   (('y', 'x'), tau,   {'units': 'Kendall tau'}),
        },
        coords={'y': y_coord, 'x': x_coord},
        attrs={
            'variable':   var_name,
            'season':     season,
            'n_min':      N_MIN,
            'years':      '1995-2025',
            'method':     'Mann-Kendall (original_test) + Sen slope',
        },
    )
    da_stack = da_stack.rio.write_crs(CRS, inplace=False)
    da_stack.rio.write_transform(transform, inplace=True)
    nc_path = TREND_DIR / f'{var_name}_{season}_trend.nc'
    da_stack.to_netcdf(nc_path, engine='netcdf4')

    # --- GeoTIFFs ---
    out_dir = TIF_DIR / var_name
    out_dir.mkdir(parents=True, exist_ok=True)

    for name, arr in [('slope', slope), ('pvalue', p), ('tau', tau)]:
        da2d = xr.DataArray(
            arr, dims=('y', 'x'),
            coords={'y': y_coord, 'x': x_coord},
            name=name,
        )
        da2d.attrs = {}
        da2d.encoding = {}
        da2d = da2d.rio.write_crs(CRS, inplace=False)
        da2d = da2d.rio.write_nodata(NODATA, inplace=False)
        da2d.rio.to_raster(
            out_dir / f'{var_name}_{season}_{name}.tif',
            dtype='float32', compress='LZW', tiled=True, nodata=NODATA,
        )
        del da2d
        gc.collect()

    # --- Significance mask (1 where p < ALPHA, 0 where not, NaN elsewhere) ---
    sig = np.where(np.isfinite(p),
                   (p < ALPHA).astype('float32'),
                   np.nan).astype('float32')
    sig_da = xr.DataArray(sig, dims=('y', 'x'),
                          coords={'y': y_coord, 'x': x_coord},
                          name='significant')
    sig_da.attrs = {}
    sig_da.encoding = {}
    sig_da = sig_da.rio.write_crs(CRS, inplace=False)
    sig_da = sig_da.rio.write_nodata(NODATA, inplace=False)
    sig_da.rio.to_raster(
        out_dir / f'{var_name}_{season}_significant.tif',
        dtype='float32', compress='LZW', tiled=True, nodata=NODATA,
    )
    del sig_da, sig
    gc.collect()

    # Return summary
    n_sig = int(np.sum((p < ALPHA) & np.isfinite(p)))
    n_valid = int(np.sum(np.isfinite(p)))
    return n_valid, n_sig, nc_path.name


# -------- Main loop --------
run_log = []

for var_name, (path, var_key) in VARIABLES.items():
    print(f"\n▶ {var_name}")
    ds = xr.open_dataset(path)
    if var_key not in ds.data_vars:
        print(f"  ✗ '{var_key}' not found in {path.name}, "
              f"available: {list(ds.data_vars)}")
        ds.close()
        continue

    da_full = ds[var_key]

    for season in SEASONS:
        print(f"  → {season}")
        mask = (da_full.season.values == season)
        if not mask.any():
            print(f"    no data for {season}, skipping")
            continue

        da_season = da_full.isel(time=mask)
        t0 = time.time()
        res = run_mk_parallel(da_season)
        elapsed = time.time() - t0

        n_valid, n_sig, fname = save_trend_outputs(
            var_name, season, res,
            da_full.y.values, da_full.x.values,
            da_full.rio.transform(),
        )

        print(f"    ✓ {fname}: valid={n_valid:,} "
              f"({100*n_valid/da_season.shape[1]/da_season.shape[2]:.1f}%), "
              f"sig={n_sig:,} ({100*n_sig/max(n_valid,1):.1f}% of valid), "
              f"t={elapsed:.1f}s")

        run_log.append({
            'variable': var_name, 'season': season,
            'valid': n_valid, 'significant': n_sig,
            'seconds': round(elapsed, 1),
        })

        del da_season, res
        gc.collect()

    ds.close()
    del da_full
    gc.collect()

log_df = pd.DataFrame(run_log)
print("\n" + "=" * 70)
print(log_df.to_string(index=False))
log_df.to_csv(LOG_DIR / 'mk_trends_log.csv', index=False)

In [ ]:
# =============================================================================
# Summary: aggregate stats + a simple preview of two trend maps.
# =============================================================================

import matplotlib.pyplot as plt

print("\nTrends written to:", TREND_DIR)
print("\nSummary per variable (across all seasons):")
agg = log_df.groupby('variable').agg(
    valid_total=('valid', 'sum'),
    sig_total=('significant', 'sum'),
    seconds=('seconds', 'sum'),
).reset_index()
agg['pct_significant'] = (100 * agg['sig_total'] / agg['valid_total']).round(1)
print(agg.to_string(index=False))

# --- Preview: NDVI JF slope + p-value ---
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
for i, season in enumerate(SEASONS):
    slope_path = TIF_DIR / 'NDVI' / f'NDVI_{season}_slope.tif'
    sig_path   = TIF_DIR / 'NDVI' / f'NDVI_{season}_significant.tif'

    slope = rxr.open_rasterio(slope_path, masked=True).squeeze()
    sig   = rxr.open_rasterio(sig_path,   masked=True).squeeze()

    axes[0, i].imshow(slope, cmap='RdBu_r', vmin=-0.01, vmax=0.01)
    axes[0, i].set_title(f'NDVI {season} slope (per year)')
    axes[0, i].axis('off')

    axes[1, i].imshow(sig, cmap='gray_r', vmin=0, vmax=1)
    axes[1, i].set_title(f'NDVI {season} significant (p<{ALPHA})')
    axes[1, i].axis('off')

plt.tight_layout()
plt.savefig(TREND_DIR / 'preview_NDVI_trends.png', dpi=110)
plt.show()